In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

pd.set_option("display.precision", 3)
plt.rcParams["figure.figsize"] = (8, 4)

# TP — Apprentissage supervisé : prédire la valeur des logements

**Master 1 - IEF - FE — Analyse de données**

Vous travaillez pour une banque qui finance des prêts immobiliers. Pour estimer rapidement la valeur des biens apportés en garantie, elle veut un modèle qui prédit la **valeur médiane des logements d'un quartier** à partir de caractéristiques simples : revenu des habitants, âge des logements, localisation…

C'est un problème de **régression supervisée** : la cible est un nombre (une valeur en dollars) et on dispose d'un historique où elle est connue.

### Les données
*California Housing Prices* (recensement américain de 1990), disponible sur le git sous le nom : `housing.csv`
Téléchargez  et placez-le dans le même dossier que ce notebook. Une ligne = un quartier (*block group*, quelques centaines à quelques milliers d'habitants).

### Consignes
- Complétez les `___` (signalés par `# TODO`).
- Les cellules **🔎 Analyse** demandent d'interpréter les résultats en une ou deux phrases : c'est la partie la plus importante du TP.

In [4]:
df = pd.read_csv("housing.csv")


df = df.rename(columns={
    "housing_median_age": "age_median", "total_rooms": "pieces_total", "total_bedrooms": "chambres_total",
    "households": "menages", "median_income": "revenu_median",
    "median_house_value": "valeur_mediane", "ocean_proximity": "proximite_ocean"})

## 1. Charger et découvrir les données

On renomme les colonnes en français pour plus de lisibilité.

| Colonne | Signification |
|---|---|
| `longitude`, `latitude` | position du quartier |
| `age_median` | âge médian des logements (années) |
| `pieces_total`, `chambres_total` | nombre total de pièces / de chambres **dans le quartier** |
| `population`, `menages` | nombre d'habitants / de ménages du quartier |
| `revenu_median` | revenu médian des ménages, **en dizaines de milliers de dollars** |
| `proximite_ocean` | catégorie : à moins d'1 h de l'océan, à l'intérieur des terres, près de l'océan, près de la baie, île |
| **`valeur_mediane`** | **cible : valeur médiane des logements du quartier (dollars)** |

**Exercice 1.1** — Affichez les premières lignes et les dimensions du tableau.

In [ ]:
# TODO : 5 premières lignes, puis dimensions
display(df.___)
print(df.___)

**Exercice 1.2** — Comptez les valeurs manquantes par colonne et affichez les statistiques descriptives.

In [ ]:
# TODO : nombre de valeurs manquantes par colonne
print(df.___)
df.describe()

**Exercice 1.3** — Tracez l'histogramme de la cible.

In [ ]:
# TODO : histogramme de la valeur médiane (50 classes)
df[___].hist(bins=___)
plt.xlabel("Valeur médiane des logements ($)"); plt.ylabel("Nombre de quartiers"); plt.show()
print("Quartiers à la valeur maximale :", (df["valeur_mediane"] == df["valeur_mediane"].max()).sum())

**Exercice 1.4** — Le lien entre revenu et valeur des logements.

> 🔎 **Analyse 1** — Que remarquez-vous à l'extrémité droite de l'histogramme ? Que s'est-il probablement passé lors de la collecte des données, et quel problème cela posera-t-il au modèle pour les quartiers les plus chers ?
>
> *Votre réponse :*

In [ ]:
# TODO : nuage de points revenu médian (x) / valeur médiane (y)
df.plot.scatter(x=___, y=___, s=2, alpha=0.2)
plt.show()
print("Corrélation :", round(df["revenu_median"].corr(df["valeur_mediane"]), 2))

> 🔎 **Analyse 2** — La relation vous paraît-elle forte ? Linéaire ? Le revenu médian suffit-il à lui seul à expliquer la valeur des logements ?
>
> *Votre réponse :*

In [ ]:
# La géographie : chaque point est un quartier, la couleur indique la valeur
df.plot.scatter(x="longitude", y="latitude", c="valeur_mediane", cmap="viridis", s=3, alpha=0.5, figsize=(7, 6))
plt.title("Valeur médiane des logements en Californie"); plt.show()

> 🔎 **Analyse 3** — Où se trouvent les quartiers les plus chers ? Une relation **linéaire** entre la longitude (ou la latitude) et la valeur peut-elle représenter cette carte ?
>
> *Votre réponse :*

## 2. Préparer les données

In [ ]:
# TODO : supprimer les lignes avec une valeur manquante
df = df.___
print("Lignes restantes :", len(df))

Les variables `pieces_total`, `chambres_total` et `population` dépendent surtout de **la taille du quartier**. On crée des variables rapportées au nombre de ménages, plus parlantes.

In [ ]:
# TODO : pièces par ménage et personnes par ménage
df["pieces_par_menage"] = df["pieces_total"] / df[___]
df["personnes_par_menage"] = df[___] / df["menages"]
df["part_chambres"] = df["chambres_total"] / df["pieces_total"]
df[["pieces_par_menage", "personnes_par_menage", "part_chambres", "valeur_mediane"]].corr()["valeur_mediane"]

> 🔎 **Analyse 4** — Pourquoi `pieces_par_menage` décrit-elle mieux un logement que `pieces_total` ? Donnez un exemple de deux quartiers qui auraient le même `pieces_total` mais des logements très différents.
>
> *Votre réponse :*

In [ ]:
# TODO : X = toutes les colonnes sauf la cible ; y = la cible
X = df.drop(columns=___)
y = df[___]
X = pd.get_dummies(X, columns=["proximite_ocean"], drop_first=True, dtype=int)

# TODO : 20 % des quartiers en test, graine 42
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=___, random_state=42)
print("Train :", X_train.shape, "| Test :", X_test.shape)

> 🔎 **Analyse 5** — Les quartiers du test vont-ils servir à entraîner les modèles ? Pourquoi les met-on de côté ?
>
> *Votre réponse :*

Pour comparer les modèles, on utilise une fonction qui calcule les métriques sur le train **et** sur le test (exécutez-la simplement) :

In [ ]:
resultats = []
def evaluer(nom, modele):
    p_train, p_test = modele.predict(X_train), modele.predict(X_test)
    res = {"modèle": nom,
           "R² train": r2_score(y_train, p_train),
           "R² test": r2_score(y_test, p_test),
           "MAE test ($)": mean_absolute_error(y_test, p_test),
           "RMSE test ($)": mean_squared_error(y_test, p_test) ** 0.5}
    resultats.append(res)
    return pd.DataFrame([res]).round(3)

In [ ]:
naif = DummyRegressor(strategy="mean")
# TODO : entraîner puis évaluer le modèle naïf
naif.fit(___)
evaluer("Naïf (moyenne)", ___)

## 3. Un repère : le modèle naïf

Le modèle naïf prédit **la même valeur pour tous les quartiers** : la moyenne du train.

> 🔎 **Analyse 6** — Que vaut le R² du modèle naïf ? Traduisez la MAE en une phrase (« en moyenne, ce modèle se trompe de … »). Tout modèle sérieux devra faire mieux que ces chiffres.
>
> *Votre réponse :*

## 4. La régression linéaire

In [ ]:
lin = LinearRegression()
# TODO : entraîner la régression linéaire et l'évaluer
lin.fit(___)
evaluer("Régression linéaire", ___)

In [ ]:
coefs = pd.Series(lin.coef_, index=X_train.columns)
coefs.round(0)

> 🔎 **Analyse 7** — Interprétez le coefficient de `revenu_median` (attention à l'unité : dizaines de milliers de dollars). Le signe et l'ordre de grandeur vous semblent-ils plausibles ?
>
> *Votre réponse :*

> 🔎 **Analyse 8** — Comparez le R² train et le R² test. Le modèle sur-apprend-il ? Selon vous, est-il plutôt limité par un **biais** ou par une **variance** trop forte ? (repensez à la carte de l'analyse 3)
>
> *Votre réponse :*

## 5. L'arbre de décision

### 5.1 Un petit arbre lisible

In [ ]:
# TODO : arbre de régression de profondeur 3
arbre3 = DecisionTreeRegressor(max_depth=___, random_state=42).fit(X_train, y_train)
plt.figure(figsize=(18, 7))
plot_tree(arbre3, feature_names=X_train.columns, filled=True, rounded=True, fontsize=9, precision=0)
plt.show()
evaluer("Arbre (profondeur 3)", arbre3)

> 🔎 **Analyse 9** — Quelle est la première question posée par l'arbre ? Que vaut la prédiction dans la feuille la plus chère, et à quels quartiers correspond-elle ? (`value` = moyenne de la cible dans la feuille)
>
> *Votre réponse :*

### 5.2 Un arbre sans limite de profondeur

In [ ]:
# TODO : arbre sans limite de profondeur (max_depth=None)
arbre_max = DecisionTreeRegressor(max_depth=___, random_state=42).fit(X_train, y_train)
print("Profondeur atteinte :", arbre_max.get_depth(), "| nombre de feuilles :", arbre_max.get_n_leaves())
evaluer("Arbre sans limite", arbre_max)

> 🔎 **Analyse 10** — Expliquez le R² train obtenu. Que vous apprend l'écart avec le R² test ?
>
> *Votre réponse :*

### 5.3 Trouver la bonne profondeur

In [ ]:
profondeurs = range(1, 21)
r2_train, r2_test = [], []
for d in profondeurs:
    # TODO : arbre de profondeur d, entraîné sur le train
    m = DecisionTreeRegressor(max_depth=___, random_state=42).fit(___)
    r2_train.append(r2_score(y_train, m.predict(X_train)))
    r2_test.append(r2_score(y_test, m.predict(X_test)))

plt.plot(profondeurs, r2_train, "o-", label="train")
plt.plot(profondeurs, r2_test, "o-", label="test")
plt.xlabel("Profondeur maximale"); plt.ylabel("R²"); plt.legend(); plt.grid(alpha=0.3); plt.show()
print("Meilleure profondeur sur le test :", list(profondeurs)[int(np.argmax(r2_test))])

> 🔎 **Analyse 11** — Décrivez les deux courbes. Où se situent le sous-apprentissage et le sur-apprentissage ? Reliez au compromis biais–variance vu en cours.
>
> *Votre réponse :*

## 6. La random forest

Une forêt moyenne les prédictions de nombreux arbres profonds, chacun entraîné sur un échantillon bootstrap. Avec `max_features=0.5`, chaque nœud ne peut choisir sa question que parmi la moitié des variables, tirée au hasard : les arbres sont ainsi plus différents les uns des autres.

In [ ]:
# TODO : forêt de 100 arbres
rf = RandomForestRegressor(n_estimators=___, max_features=___, min_samples_leaf=2, n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)
evaluer("Random forest", rf)

In [ ]:
pd.DataFrame(resultats).drop_duplicates("modèle", keep="last").set_index("modèle").round(3)

> 🔎 **Analyse 12** — Classez les modèles selon le R² test et selon la MAE. De combien de dollars la forêt réduit-elle l'erreur moyenne par rapport à la régression linéaire ? Pour la banque, est-ce un gain important ?
>
> *Votre réponse :*

> 🔎 **Analyse 13** — La forêt a un R² train bien plus élevé que son R² test. Est-ce un problème, sachant que son R² test est le meilleur de tous ? Sur quel chiffre doit-on juger un modèle ?
>
> *Votre réponse :*

### 6.1 Combien d'arbres ?

In [ ]:
n_arbres = [1, 5, 10, 25, 50, 100]
r2s = []
for B in n_arbres:
    # TODO : forêt de B arbres
    m = RandomForestRegressor(n_estimators=___, max_features=0.5, min_samples_leaf=2, n_jobs=-1, random_state=42).fit(X_train, y_train)
    r2s.append(r2_score(y_test, m.predict(X_test)))
plt.plot(n_arbres, r2s, "o-"); plt.xscale("log"); plt.xlabel("Nombre d'arbres"); plt.ylabel("R² test"); plt.grid(alpha=0.3); plt.show()

> 🔎 **Analyse 14** — La performance continue-t-elle d'augmenter indéfiniment ? Ajouter des arbres fait-il sur-apprendre ? Notez que la forêt à 1 arbre est simplement un arbre profond.
>
> *Votre réponse :*

### 6.2 Quelles variables comptent ?

Importance par permutation : on mélange une variable dans le test et on mesure la **baisse de R²**. Plus elle est forte, plus le modèle dépend de cette variable.

In [ ]:
# TODO : importance par permutation sur le test (critère R², 5 répétitions)
perm = permutation_importance(rf, ___, scoring="r2", n_repeats=5, random_state=42, n_jobs=-1)
imp = pd.Series(perm.importances_mean, index=X_test.columns).sort_values()
imp.plot.barh(title="Baisse de R² quand on mélange la variable"); plt.show()

> 🔎 **Analyse 15** — Quelles sont les trois variables les plus importantes ? Pourquoi la longitude et la latitude sont-elles utiles à la forêt, alors que leurs coefficients linéaires étaient difficiles à interpréter ?
>
> *Votre réponse :*

> 🔎 **Analyse 16** — Peut-on conclure qu'augmenter le revenu des habitants d'un quartier **ferait monter** la valeur de ses logements ? (prédiction ou causalité ?)
>
> *Votre réponse :*

## Synthèse *(à compléter avant de partir)*

| Modèle | R² test | MAE test ($) | Interprétable ? | Capte la géographie et les non-linéarités ? |
|---|---|---|---|---|
| Naïf | | | — | — |
| Régression linéaire | | | | |
| Arbre (profondeur 3) | | | | |
| Random forest | | | | |

> 🔎 **Analyse finale** — Quel modèle recommanderiez-vous à la banque ? Donnez un argument pour et un argument contre.
>
> *Votre réponse :*